# 10. Calidad de señal y minutos evaluables

## Objetivo

Evaluar la política de exclusión de minutos antes de definir los
umbrales de clasificación de apnea.

La política vigente exige una etiqueta válida, una característica
calculable y una proporción de intervalos RR corregidos no mayor al 20 %.

Se evaluará también la regla candidata fijada en el plan: excluir los
minutos cuya ventana de análisis de 240 segundos contenga señal perdida,
identificada por tramos planos o clavados en el tope del conversor
durante al menos un segundo.

Se analizarán:
- La cantidad de minutos y registros afectados.
- La diferencia de exclusión entre minutos con apnea y normales,
  con intervalos de confianza por sujeto.
- El desempeño de F y la cobertura de cada política.

Se utilizan exclusivamente los registros de desarrollo, con las
marcas QRS de la base y sin Wiener. Las decisiones se documentarán
después de obtener los resultados.

In [1]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import os
import sys

ROOT = Path("/content/drive/MyDrive/TP-PSB-G8-main")
assert (ROOT / "src").is_dir(), f"No se encontró el proyecto en {ROOT}"

os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

%pip install -q -r requirements.txt

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.config import SPLIT_CSV, RAW_DIR
from src.data import load_ecg_digital, load_header, load_apnea_labels
from src.preprocessing.quality import flat_mask, saturated_mask

split = pd.read_csv(SPLIT_CSV)
DEV = split.loc[split["set"].eq("dev"), "record"].tolist()

CACHE_CALIDAD = ROOT / "results" / "calidad" / "v1"
CACHE_CALIDAD.mkdir(parents=True, exist_ok=True)

faltantes = [
    f"{rec}{ext}"
    for rec in DEV
    for ext in (".dat", ".hea", ".qrs")
    if not (RAW_DIR / f"{rec}{ext}").exists()
]

print(f"Registros de desarrollo: {len(DEV)}")
print(f"Archivos de ECG/QRS faltantes: {len(faltantes)}")
print(f"Tablas previas de calidad: {len(list(CACHE_CALIDAD.glob('*.parquet')))}")

if faltantes:
    print("Primeros faltantes:", faltantes[:10])

assert len(DEV) == 47, "La partición no coincide con los 47 registros esperados."
assert not faltantes, "Faltan archivos; resolver esto antes de continuar."

Mounted at /content/drive
Registros de desarrollo: 47
Archivos de ECG/QRS faltantes: 0
Tablas previas de calidad: 0


## Señal perdida en la ventana de análisis

Se analiza el ECG en unidades digitales, antes del filtrado, para
identificar valores exactamente constantes y los topes del conversor.

Se considera señal perdida:
- Un tramo constante de al menos un segundo.
- Un tramo que permanece en los topes del conversor durante al menos
  un segundo.

Para cada minuto se examina su ventana centrada de 240 segundos.
Se registra si contiene señal perdida y su duración dentro de la ventana.
Las ventanas incompletas en los extremos de la grabación se identifican
por separado.

Este criterio identifica pérdida de señal; no detecta todos los tipos
de ruido.

In [2]:
def calidad_por_ventana(rec):
    """Señal perdida en ventanas de 240 s centradas en cada minuto."""
    d = load_ecg_digital(rec)
    header = load_header(rec)
    fs = header["fs"]
    labels = load_apnea_labels(rec)

    # Tramos constantes de al menos 1 s.
    planos = flat_mask(d, fs, min_s=1.0)

    # Permanencia en los topes durante al menos 1 s.
    en_tope = saturated_mask(d - header["cero"], header["bits"])
    saturados_largos = en_tope & flat_mask(
        en_tope.astype(np.int8), fs, min_s=1.0
    )

    perdida = planos | saturados_largos

    minutos = np.arange(len(labels))
    inicio_s = 60 * minutos + 30 - 120
    fin_s = inicio_s + 240

    inicio = np.rint(inicio_s * fs).astype(int)
    fin = np.rint(fin_s * fs).astype(int)
    completa = (inicio >= 0) & (fin <= len(d))

    # Conteo acumulado: evita recorrer nuevamente cada ventana.
    acumulado = np.empty(len(d) + 1, dtype=np.int64)
    acumulado[0] = 0
    np.cumsum(perdida, dtype=np.int64, out=acumulado[1:])

    segundos_perdidos = np.full(len(minutos), np.nan)
    segundos_perdidos[completa] = (
        acumulado[fin[completa]] - acumulado[inicio[completa]]
    ) / fs

    return pd.DataFrame({
        "record": rec,
        "minute": minutos,
        "label": labels,
        "ventana_completa": completa,
        "segundos_perdidos": segundos_perdidos,
        "sin_perdida": completa & (segundos_perdidos == 0),
    })


tablas = []

for i, rec in enumerate(DEV, 1):
    ruta = CACHE_CALIDAD / f"{rec}.parquet"

    if ruta.exists():
        tabla = pd.read_parquet(ruta)
        estado = "guardado"
    else:
        tabla = calidad_por_ventana(rec)
        tabla.to_parquet(ruta, index=False)
        estado = "calculado"

    tablas.append(tabla)
    print(f"{i}/{len(DEV)} — {rec}: {estado}", flush=True)

calidad = pd.concat(tablas, ignore_index=True)

# Resumen sobre minutos etiquetados con ventana completa.
evaluables = calidad.loc[
    calidad["label"].isin([0, 1]) & calidad["ventana_completa"]
].copy()

resumen_calidad = (
    evaluables.groupby("record")
    .agg(
        minutos=("minute", "size"),
        sin_perdida=("sin_perdida", "sum"),
    )
)

resumen_calidad["con_perdida"] = (
    resumen_calidad["minutos"] - resumen_calidad["sin_perdida"]
)
resumen_calidad["porcentaje_con_perdida"] = (
    100 * resumen_calidad["con_perdida"] / resumen_calidad["minutos"]
)

display(
    resumen_calidad.sort_values(
        "porcentaje_con_perdida", ascending=False
    ).head(10).round(2)
)

print(f"\nMinutos con ventana completa: {len(evaluables)}")
print(f"Ventanas con señal perdida: {(~evaluables['sin_perdida']).sum()}")
print(
    "Registros con alguna ventana afectada:",
    int((resumen_calidad["con_perdida"] > 0).sum()),
)

1/47 — a01: calculado
2/47 — a02: calculado
3/47 — a03: calculado
4/47 — a04: calculado
5/47 — a05: calculado
6/47 — a08: calculado
7/47 — a09: calculado
8/47 — a10: calculado
9/47 — a12: calculado
10/47 — a13: calculado
11/47 — a14: calculado
12/47 — a15: calculado
13/47 — a17: calculado
14/47 — a18: calculado
15/47 — a19: calculado
16/47 — a20: calculado
17/47 — b01: calculado
18/47 — b02: calculado
19/47 — b03: calculado
20/47 — c01: calculado
21/47 — c02: calculado
22/47 — c03: calculado
23/47 — c06: calculado
24/47 — c07: calculado
25/47 — c09: calculado
26/47 — c10: calculado
27/47 — x03: calculado
28/47 — x04: calculado
29/47 — x05: calculado
30/47 — x07: calculado
31/47 — x08: calculado
32/47 — x09: calculado
33/47 — x10: calculado
34/47 — x12: calculado
35/47 — x14: calculado
36/47 — x16: calculado
37/47 — x17: calculado
38/47 — x18: calculado
39/47 — x19: calculado
40/47 — x20: calculado
41/47 — x21: calculado
42/47 — x23: calculado
43/47 — x25: calculado
44/47 — x27: calcula

,minutos,sin_perdida,con_perdida,porcentaje_con_perdida
record,,,,
a17,480,436,44,9.17
a15,505,473,32,6.34
x20,508,481,27,5.31
x18,455,440,15,3.30
a19,498,482,16,3.21
x10,506,490,16,3.16
c09,464,451,13,2.80
c10,427,417,10,2.34
x35,478,469,9,1.88



Minutos con ventana completa: 22960
Ventanas con señal perdida: 240
Registros con alguna ventana afectada: 20


In [3]:
# Reutilizamos F y la proporción de RR corregidos del análisis QRS.
# Las columnas "base" corresponden a las marcas de la base, sin Wiener.
ruta_f = (
    ROOT / "results" / "qrs" / "pan_tompkins_v1"
    / "comparacion_F_v1"
)

faltantes_f = [
    rec for rec in DEV
    if not (ruta_f / f"{rec}.parquet").exists()
]
assert not faltantes_f, (
    f"No se encontraron las tablas previas de estos registros: {faltantes_f}"
)

tablas_f = []
for rec in DEV:
    t = pd.read_parquet(ruta_f / f"{rec}.parquet")
    tablas_f.append(
        t[["record", "minute", "label_base", "corrected_base", "F_base"]]
    )

features_base = pd.concat(tablas_f, ignore_index=True)

analisis = calidad.merge(
    features_base,
    on=["record", "minute"],
    how="left",
    validate="one_to_one",
)

# Verificamos que las etiquetas coincidan entre los dos cálculos.
con_etiqueta_previa = analisis["label_base"].notna()
assert (
    analisis.loc[con_etiqueta_previa, "label"]
    == analisis.loc[con_etiqueta_previa, "label_base"]
).all(), "Las etiquetas no coinciden."

# Universo de comparación: minutos etiquetados, ventana completa y F finita.
calculable = (
    analisis["label"].isin([0, 1])
    & analisis["ventana_completa"]
    & np.isfinite(analisis["F_base"])
    & np.isfinite(analisis["corrected_base"])
)

analisis = analisis.loc[calculable].copy()

analisis["regla_20"] = analisis["corrected_base"] <= 0.20
analisis["regla_20_sin_perdida"] = (
    analisis["regla_20"] & analisis["sin_perdida"]
)

politicas = {
    "F calculable, sin exclusión por calidad": np.ones(
        len(analisis), dtype=bool
    ),
    "Regla actual: RR corregidos ≤ 20 %": analisis["regla_20"],
    "Regla candidata: actual + sin señal perdida":
        analisis["regla_20_sin_perdida"],
}

filas = []
for nombre, conservar in politicas.items():
    conservados = analisis.loc[conservar]
    filas.append({
        "Política": nombre,
        "Minutos conservados": len(conservados),
        "Minutos excluidos": len(analisis) - len(conservados),
        "Registros conservados": conservados["record"].nunique(),
        "Porcentaje conservado": 100 * len(conservados) / len(analisis),
    })

display(pd.DataFrame(filas).set_index("Política").round(2))

# Pérdidas adicionales respecto de la política vigente.
vigentes = analisis.loc[analisis["regla_20"]].copy()
vigentes["exclusion_adicional"] = ~vigentes["sin_perdida"]

por_clase = (
    vigentes.groupby("label")
    .agg(
        minutos_actuales=("minute", "size"),
        exclusiones_adicionales=("exclusion_adicional", "sum"),
    )
    .reindex([0, 1])
)
por_clase["porcentaje_excluido_adicional"] = (
    100 * por_clase["exclusiones_adicionales"]
    / por_clase["minutos_actuales"]
)
por_clase.index = ["Normal", "Apnea"]

display(por_clase.round(2))

print(
    "Ventanas con pérdida que la regla del 20 % ya excluía:",
    int((~analisis["sin_perdida"] & ~analisis["regla_20"]).sum()),
)
print(
    "Ventanas que se excluirían adicionalmente:",
    int(vigentes["exclusion_adicional"].sum()),
)

,Minutos conservados,Minutos excluidos,Registros conservados,Porcentaje conservado
Política,,,,
"F calculable, sin exclusión por calidad",22890,0,47,100.00
Regla actual: RR corregidos ≤ 20 %,22275,615,47,97.31
Regla candidata: actual + sin señal perdida,22115,775,47,96.61


,minutos_actuales,exclusiones_adicionales,porcentaje_excluido_adicional
Normal,13421,91,0.68
Apnea,8854,69,0.78


Ventanas con pérdida que la regla del 20 % ya excluía: 33
Ventanas que se excluirían adicionalmente: 160


In [4]:
# Conteos por sujeto y clase entre los minutos de la política vigente.
datos_sesgo = vigentes.merge(
    split[["record", "subject"]],
    on="record",
    how="left",
    validate="many_to_one",
)
assert datos_sesgo["subject"].notna().all()

sujetos = sorted(datos_sesgo["subject"].unique())

conteos = (
    datos_sesgo.groupby(["subject", "label"])
    .agg(
        total=("minute", "size"),
        excluidos=("exclusion_adicional", "sum"),
    )
    .reindex(
        pd.MultiIndex.from_product(
            [sujetos, [0, 1]], names=["subject", "label"]
        ),
        fill_value=0,
    )
)

# Filas: sujetos. Columnas: normal (0), apnea (1).
totales = conteos["total"].to_numpy().reshape(-1, 2)
excluidos = conteos["excluidos"].to_numpy().reshape(-1, 2)

tasas = excluidos.sum(axis=0) / totales.sum(axis=0)
diferencia = 100 * (tasas[1] - tasas[0])

rng = np.random.default_rng(2026)
diferencias_boot = []

for _ in range(2000):
    seleccion = rng.integers(0, len(sujetos), size=len(sujetos))
    n = totales[seleccion].sum(axis=0)
    e = excluidos[seleccion].sum(axis=0)

    if (n > 0).all():
        tasas_boot = e / n
        diferencias_boot.append(
            100 * (tasas_boot[1] - tasas_boot[0])
        )

ic_lo, ic_hi = np.percentile(diferencias_boot, [2.5, 97.5])

resultado_sesgo = pd.DataFrame([{
    "Sujetos": len(sujetos),
    "Exclusión normal (%)": 100 * tasas[0],
    "Exclusión apnea (%)": 100 * tasas[1],
    "Diferencia apnea − normal (puntos porcentuales)": diferencia,
    "IC95 inferior": ic_lo,
    "IC95 superior": ic_hi,
}])

display(resultado_sesgo.round(3))

,Sujetos,Exclusión normal (%),Exclusión apnea (%),Diferencia apnea − normal (puntos porcentuales),IC95 inferior,IC95 superior
0,19,0.678,0.779,0.101,-0.474,0.679


In [5]:
from sklearn.metrics import roc_auc_score, roc_curve
from scipy.stats import spearmanr

datos = datos_sesgo.copy().reset_index(drop=True)

y = datos["label"].to_numpy(dtype=int)
f = datos["F_base"].to_numpy(dtype=float)
conservar = datos["sin_perdida"].to_numpy(dtype=bool)

# Un mismo umbral para ambas políticas.
fpr, tpr, umbrales = roc_curve(y, f)
validos = np.isfinite(umbrales)
umbral_control = umbrales[validos][
    np.argmax((tpr - fpr)[validos])
]

datos["prediccion"] = (f >= umbral_control).astype(int)

# Fracciones de apnea reales y estimadas en los minutos de cada política.
def resumen_registros(tabla):
    return (
        tabla.groupby(["record", "subject"])
        .agg(
            real=("label", "mean"),
            estimada=("prediccion", "mean"),
            minutos=("minute", "size"),
        )
    )

actual = resumen_registros(datos)
candidata = resumen_registros(datos.loc[conservar])

assert actual.index.equals(candidata.index), (
    "Algún registro quedó sin minutos en la política candidata."
)

auc_actual = roc_auc_score(y, f)
auc_candidata = roc_auc_score(y[conservar], f[conservar])

rho_actual = spearmanr(actual["real"], actual["estimada"]).statistic
rho_candidata = spearmanr(
    candidata["real"], candidata["estimada"]
).statistic

# Bootstrap pareado por sujeto: cada remuestreo se usa en ambas políticas.
sujetos_control = sorted(datos["subject"].unique())
indice_sujeto = {s: i for i, s in enumerate(sujetos_control)}

sujeto_minuto = datos["subject"].map(indice_sujeto).to_numpy()
sujeto_registro = np.array([
    indice_sujeto[s]
    for s in actual.index.get_level_values("subject")
])

rng = np.random.default_rng(2026)
deltas_auc = []
deltas_rho = []

for _ in range(2000):
    seleccion = rng.integers(
        0, len(sujetos_control), size=len(sujetos_control)
    )
    multiplicidad = np.bincount(
        seleccion, minlength=len(sujetos_control)
    )

    pesos = multiplicidad[sujeto_minuto]

    # AUC: reproducimos la repetición de sujetos mediante pesos.
    if (
        pesos[y == 0].sum() > 0
        and pesos[y == 1].sum() > 0
        and pesos[conservar & (y == 0)].sum() > 0
        and pesos[conservar & (y == 1)].sum() > 0
    ):
        a = roc_auc_score(y, f, sample_weight=pesos)
        c = roc_auc_score(
            y[conservar], f[conservar],
            sample_weight=pesos[conservar],
        )
        deltas_auc.append(c - a)

    # Spearman: repetimos las filas de los registros del sujeto elegido.
    filas = np.repeat(
        np.arange(len(actual)),
        multiplicidad[sujeto_registro],
    )
    a = spearmanr(
        actual["real"].to_numpy()[filas],
        actual["estimada"].to_numpy()[filas],
    ).statistic
    c = spearmanr(
        candidata["real"].to_numpy()[filas],
        candidata["estimada"].to_numpy()[filas],
    ).statistic

    if np.isfinite(a) and np.isfinite(c):
        deltas_rho.append(c - a)

filas_resultado = []

for nombre, base, nueva, diferencias in [
    ("AUC por minuto", auc_actual, auc_candidata, deltas_auc),
    ("Spearman por registro", rho_actual, rho_candidata, deltas_rho),
]:
    lo, hi = np.percentile(diferencias, [2.5, 97.5])
    filas_resultado.append({
        "Métrica": nombre,
        "Actual": base,
        "Candidata": nueva,
        "Diferencia": nueva - base,
        "IC95 inferior": lo,
        "IC95 superior": hi,
        "Límite inferior ≥ −0,02": lo >= -0.02,
    })

comparacion_calidad = pd.DataFrame(filas_resultado).set_index("Métrica")

print(f"Umbral de control compartido: {umbral_control:.4f}")
display(comparacion_calidad.round(4))

Umbral de control compartido: -6.9486


,Actual,Candidata,Diferencia,IC95 inferior,IC95 superior,"Límite inferior ≥ −0,02"
Métrica,,,,,,
AUC por minuto,0.8733,0.8771,0.0038,0.0014,0.0079,True
Spearman por registro,0.8134,0.8119,-0.0015,-0.0209,0.0109,False


## Resultados y decisión

La política actual conserva 22.275 minutos de los 47 registros de
desarrollo. Agregar la exclusión de ventanas con señal perdida
conservaría 22.115 minutos: se excluirían 160 minutos adicionales
(0,72 % de los actualmente evaluados), sin perder registros completos.

La exclusión adicional fue del 0,678 % en minutos normales y del
0,779 % en minutos con apnea. La diferencia fue de 0,101 puntos
porcentuales, con IC95 % por sujeto de −0,474 a +0,679. No se
identificó una diferencia clara de exclusión entre las clases.

Con un umbral de control compartido de −6,9486:
- El AUC pasó de 0,8733 a 0,8771. Diferencia: +0,0038;
  IC95 %: +0,0014 a +0,0079.
- Spearman por registro pasó de 0,8134 a 0,8119.
  Diferencia: −0,0015; IC95 %: −0,0209 a +0,0109.

La regla candidata no cumplió el criterio de no inferioridad en
ambas métricas, porque el límite inferior de Spearman quedó por
debajo de −0,02. Se mantiene la política vigente: etiqueta válida,
F calculable y no más del 20 % de RR corregidos.

Esta comparación describe el efecto de excluir minutos; no modifica
F ni demuestra una mejora del detector sobre una población fija.
El umbral utilizado es diagnóstico y no constituye el umbral final.

Como limitación, la política vigente deja pasar algunos minutos
con señal perdida. Además, este análisis no valida individualmente
los límites empleados para corregir los intervalos RR.

In [6]:
codigo_calidad = '''

# --- Calidad por ventana: notebook 10 ---
def window_signal_quality(d, fs, bits, minutes, width_s=240.0, adc_zero=0):
    """Calidad del ECG digital en ventanas centradas en cada minuto.

    Señal perdida: tramo plano o permanencia en los topes durante
    al menos un segundo. Las ventanas incompletas no son evaluables.
    """
    import pandas as pd

    d = np.asarray(d)
    minutes = np.asarray(minutes, dtype=int)

    if d.ndim != 1 or d.size == 0:
        raise ValueError("El ECG debe ser un vector no vacío.")
    if not np.isfinite(fs) or fs <= 0 or width_s <= 0:
        raise ValueError("fs y width_s deben ser positivos.")

    planos = flat_mask(d, fs, min_s=1.0)
    en_tope = saturated_mask(d - adc_zero, bits)
    saturados_largos = en_tope & flat_mask(
        en_tope.astype(np.int8), fs, min_s=1.0
    )
    perdida = planos | saturados_largos

    inicio = np.rint(
        (60 * minutes + 30 - width_s / 2) * fs
    ).astype(int)
    fin = np.rint(
        (60 * minutes + 30 + width_s / 2) * fs
    ).astype(int)
    completa = (inicio >= 0) & (fin <= len(d))

    acumulado = np.r_[0, np.cumsum(perdida, dtype=np.int64)]
    segundos = np.full(len(minutes), np.nan)
    segundos[completa] = (
        acumulado[fin[completa]] - acumulado[inicio[completa]]
    ) / fs

    return pd.DataFrame({
        "minute": minutes,
        "ventana_completa": completa,
        "segundos_perdidos": segundos,
        "sin_perdida": completa & (segundos == 0),
    })
'''

ruta = ROOT / "src" / "preprocessing" / "quality.py"
contenido = ruta.read_text()

if "def window_signal_quality(" not in contenido:
    ruta.write_text(contenido + codigo_calidad)
    print("Función agregada a quality.py")
else:
    print("La función ya existe; no se agregó nuevamente.")

Función agregada a quality.py


In [7]:
codigo_build = '''

# --- Política de calidad: notebook 10 ---
def attach_window_quality(tab, rec, width_s=240.0,
                          raw_dir=RAW_DIR, exclude_lost=False):
    """Adjunta calidad del ECG a una tabla por minuto.

    Por defecto conserva todas las filas. exclude_lost=True aplica
    la regla candidata evaluada y no adoptada en el notebook 10.
    No modifica F ni aplica la regla de RR corregidos.
    """
    from ..data import load_ecg_digital, load_header
    from ..preprocessing.quality import window_signal_quality

    d = load_ecg_digital(rec, raw_dir)
    header = load_header(rec, raw_dir)
    calidad = window_signal_quality(
        d, header["fs"], header["bits"],
        tab["minute"].to_numpy(),
        width_s=width_s,
        adc_zero=header["cero"],
    )

    out = tab.copy()
    for col in ("ventana_completa", "segundos_perdidos", "sin_perdida"):
        out[col] = calidad[col].to_numpy()

    if exclude_lost:
        out = out.loc[out["sin_perdida"]].copy()
    return out
'''

ruta = ROOT / "src" / "features" / "build.py"
contenido = ruta.read_text()

if "def attach_window_quality(" not in contenido:
    ruta.write_text(contenido + codigo_build)
    print("Opción de calidad agregada a build.py")
else:
    print("La función ya existe; no se agregó nuevamente.")

Opción de calidad agregada a build.py


In [8]:
codigo_tests = '''
import numpy as np
import pandas as pd

from src.preprocessing.quality import window_signal_quality


def signal():
    # Señal variable, sin tramos planos ni saturación.
    return np.tile([-2, 2], 3000)


def test_ventanas_completas_y_sin_perdida():
    q = window_signal_quality(signal(), 10, 12, [0, 2])
    assert not q.loc[0, "ventana_completa"]
    assert np.isnan(q.loc[0, "segundos_perdidos"])
    assert not q.loc[0, "sin_perdida"]
    assert q.loc[1, "sin_perdida"]


def test_duracion_minima_y_limites():
    d = signal()
    d[1000:1009] = 0  # 0,9 s: no alcanza el mínimo.
    q = window_signal_quality(d, 10, 12, [2])
    assert q.loc[0, "sin_perdida"]

    d = signal()
    d[1000:1010] = 0  # 1 s dentro de la ventana.
    q = window_signal_quality(d, 10, 12, [2])
    assert q.loc[0, "segundos_perdidos"] == 1.0

    d = signal()
    d[290:300] = 0  # Termina justo donde empieza la ventana.
    q = window_signal_quality(d, 10, 12, [2])
    assert q.loc[0, "sin_perdida"]


def test_permanencia_en_topes():
    d = signal()
    # Alterna entre topes: no es plano, pero permanece saturado 1 s.
    d[1000:1010] = np.tile([-2048, 2047], 5)
    q = window_signal_quality(d, 10, 12, [2])
    assert q.loc[0, "segundos_perdidos"] == 1.0
    assert not q.loc[0, "sin_perdida"]


def test_exclusion_es_opcional(monkeypatch):
    import src.data as data
    from src.features.build import attach_window_quality

    d = signal()
    d[1000:1010] = 0

    monkeypatch.setattr(data, "load_ecg_digital", lambda *args: d)
    monkeypatch.setattr(
        data, "load_header",
        lambda *args: {"fs": 10, "bits": 12, "cero": 0},
    )

    tab = pd.DataFrame({"minute": [2, 6], "F": [1.0, 2.0]})
    actual = attach_window_quality(tab, "sintetico")
    candidata = attach_window_quality(
        tab, "sintetico", exclude_lost=True
    )

    pd.testing.assert_frame_equal(actual[tab.columns], tab)
    assert candidata["minute"].tolist() == [6]
'''

(ROOT / "tests" / "test_quality.py").write_text(codigo_tests)

# Proceso nuevo: carga las funciones recién guardadas.
!python -m pytest tests/test_quality.py -q

....                                                                     [100%]
4 passed in 6.29s


In [9]:
import importlib
import src.preprocessing.quality as quality_module

importlib.reload(quality_module)

for i, rec in enumerate(DEV, 1):
    guardada = pd.read_parquet(CACHE_CALIDAD / f"{rec}.parquet")
    header = load_header(rec)

    nueva = quality_module.window_signal_quality(
        load_ecg_digital(rec),
        header["fs"],
        header["bits"],
        guardada["minute"].to_numpy(),
        width_s=240.0,
        adc_zero=header["cero"],
    )

    columnas = [
        "minute", "ventana_completa",
        "segundos_perdidos", "sin_perdida",
    ]
    pd.testing.assert_frame_equal(
        guardada[columnas].reset_index(drop=True),
        nueva[columnas].reset_index(drop=True),
    )

    print(f"{i}/{len(DEV)} — {rec}: coincide", flush=True)

print("\nVerificación completa: las 47 tablas coinciden.")

1/47 — a01: coincide
2/47 — a02: coincide
3/47 — a03: coincide
4/47 — a04: coincide
5/47 — a05: coincide
6/47 — a08: coincide
7/47 — a09: coincide
8/47 — a10: coincide
9/47 — a12: coincide
10/47 — a13: coincide
11/47 — a14: coincide
12/47 — a15: coincide
13/47 — a17: coincide
14/47 — a18: coincide
15/47 — a19: coincide
16/47 — a20: coincide
17/47 — b01: coincide
18/47 — b02: coincide
19/47 — b03: coincide
20/47 — c01: coincide
21/47 — c02: coincide
22/47 — c03: coincide
23/47 — c06: coincide
24/47 — c07: coincide
25/47 — c09: coincide
26/47 — c10: coincide
27/47 — x03: coincide
28/47 — x04: coincide
29/47 — x05: coincide
30/47 — x07: coincide
31/47 — x08: coincide
32/47 — x09: coincide
33/47 — x10: coincide
34/47 — x12: coincide
35/47 — x14: coincide
36/47 — x16: coincide
37/47 — x17: coincide
38/47 — x18: coincide
39/47 — x19: coincide
40/47 — x20: coincide
41/47 — x21: coincide
42/47 — x23: coincide
43/47 — x25: coincide
44/47 — x27: coincide
45/47 — x28: coincide
46/47 — x34: coinci

In [10]:
!python -m pytest -q

.................sss.............................................x...... [ 80%]
............s....                                                        [100%]
84 passed, 4 skipped, 1 xfailed in 64.41s (0:01:04)
